# PAUL Open Model - End-to-End Orchestrated Pipeline


In [ ]:
# PHASE 0: Environment + Authentication
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/PAUL_Open_Model'
else:
    print("Not running in Colab. Using local directory...")
    BASE_DIR = './'

if IN_COLAB:
    !pip install -q -U transformers trl peft bitsandbytes accelerate datasets

print("Environment setup complete.")


In [ ]:
# State Management and Idempotency
import json
import uuid
import time
from pathlib import Path

STATE_DIR = os.path.join(BASE_DIR, 'manifests', 'state')
os.makedirs(STATE_DIR, exist_ok=True)

run_id_file = os.path.join(STATE_DIR, 'current_run_id.txt')
if os.path.exists(run_id_file):
    with open(run_id_file, 'r') as f:
        RUN_ID = f.read().strip()
    print(f"Resuming existing Run ID: {RUN_ID}")
else:
    RUN_ID = f"paul_gemma4_e4b_{uuid.uuid4().hex[:8]}"
    with open(run_id_file, 'w') as f:
        f.write(RUN_ID)
    print(f"Created new Run ID: {RUN_ID}")

state_file = os.path.join(STATE_DIR, f"{RUN_ID}_state.json")

def load_state():
    if os.path.exists(state_file):
        with open(state_file, 'r') as f:
            return json.load(f)
    return {"CURRENT_PHASE": "INIT", "AUTHORIZATIONS": []}

def save_state(state):
    with open(state_file, 'w') as f:
        json.dump(state, f, indent=2)

def consume_authorization(auth_token):
    state = load_state()
    if auth_token in state.get("AUTHORIZATIONS", []):
        return True
    return False

def add_authorization(auth_token):
    state = load_state()
    if "AUTHORIZATIONS" not in state:
        state["AUTHORIZATIONS"] = []
    if auth_token not in state["AUTHORIZATIONS"]:
        state["AUTHORIZATIONS"].append(auth_token)
    save_state(state)

def set_phase(phase):
    state = load_state()
    state["CURRENT_PHASE"] = phase
    save_state(state)
    print(f"STATUS: {phase}")

state = load_state()
print(f"Current Phase: {state['CURRENT_PHASE']}")


In [ ]:
# PHASE 1: Drive + Dataset Verification
import hashlib

DATA_DIR = os.path.join(BASE_DIR, 'data', 'train') if IN_COLAB else os.path.join(BASE_DIR, 'data')

SFT_PATH = os.path.join(DATA_DIR, 'sft_train.jsonl')
DPO_PATH = os.path.join(DATA_DIR, 'dpo_train.jsonl')
MANIFEST_PATH = os.path.join(DATA_DIR, 'generation_progress_v2.json')

EXPECTED_SFT_HASH = "4c48d5077a5a6df0da7fed592c17dfd00f172da0f4a00ece0c7b682d7e2ef875"
EXPECTED_DPO_HASH = "a613e476cb161ffaddd1638bbd302d8b63c4534e4dabc33936756278cebd245e"
EXPECTED_MANIFEST_HASH = "36414ac115f075bc8845f274303705c806e33962529624a1d567d774f9473caa"

def get_hash(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()

def count_records(path):
    with open(path, 'r') as f:
        return sum(1 for _ in f)

print("Verifying SFT...")
sft_hash = get_hash(SFT_PATH)
sft_count = count_records(SFT_PATH)
if sft_hash != EXPECTED_SFT_HASH or sft_count != 180:
    raise RuntimeError(f"SFT Dataset integrity failure! Hash: {sft_hash}, Count: {sft_count}")

print("Verifying DPO...")
dpo_hash = get_hash(DPO_PATH)
dpo_count = count_records(DPO_PATH)
if dpo_hash != EXPECTED_DPO_HASH or dpo_count != 65:
    raise RuntimeError(f"DPO Dataset integrity failure! Hash: {dpo_hash}, Count: {dpo_count}")

print("Verifying Manifest...")
manifest_hash = get_hash(MANIFEST_PATH)
if manifest_hash != EXPECTED_MANIFEST_HASH:
    raise RuntimeError("Manifest integrity failure!")

print("All datasets verified successfully. Total records: 245.")


In [ ]:
# PHASE 2: T4 Model Load
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. STOP.")

gpu_name = torch.cuda.get_device_name(0)
if "T4" not in gpu_name:
    raise RuntimeError(f"Hardware mismatch. Required: Tesla T4, Found: {gpu_name}")

free_memory, total_memory = torch.cuda.mem_get_info(0)
allocated_before = torch.cuda.memory_allocated(0)
reserved_before = torch.cuda.memory_reserved(0)

print(f"GPU: {gpu_name}")
print(f"Total VRAM: {total_memory / 1024**3:.2f} GB")
print(f"Free VRAM before load: {free_memory / 1024**3:.2f} GB")
print(f"Allocated VRAM before load: {allocated_before / 1024**3:.2f} GB")

BASE_MODEL_ID = "google/gemma-4-E4B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading {BASE_MODEL_ID} in 4-bit NF4 with double quantization and FP16 compute...")

try:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
except Exception as e:
    set_phase("T4_MODEL_LOAD_RUNTIME_FAIL")
    raise RuntimeError(f"Model load failed: {str(e)}")

allocated_after = torch.cuda.memory_allocated(0)
reserved_after = torch.cuda.memory_reserved(0)
peak_allocated = torch.cuda.max_memory_allocated(0)
peak_reserved = torch.cuda.max_memory_reserved(0)

print(f"Allocated VRAM after load: {allocated_after / 1024**3:.2f} GB")
print(f"Peak Allocated VRAM: {peak_allocated / 1024**3:.2f} GB")


In [ ]:
# PHASE 3: PEFT Attachment
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# PHASE 4: Forward Dry Run
import time
from datasets import load_dataset

sft_dataset = load_dataset("json", data_files=SFT_PATH, split="train")

def format_sft(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "model", "content": example["completion"]}
    ]
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example

sft_dataset = sft_dataset.map(format_sft)

print("Running forward pass dry run...")
inputs = tokenizer(sft_dataset[0]['text'], return_tensors='pt', padding=True, truncation=True, max_length=4096).to("cuda")

start_time = time.time()
with torch.no_grad():
    outputs = model(**inputs)
execution_time = time.time() - start_time

print(f"Forward pass successful. Execution time: {execution_time:.2f}s")
print(f"Logits shape: {outputs.logits.shape}")

set_phase("AWAITING_SFT_AUTHORIZATION")
print("STATUS: T4_DRY_RUN_PASS")

report = {
    "gpu": gpu_name,
    "total_vram_gb": total_memory / 1024**3,
    "peak_vram_gb": torch.cuda.max_memory_allocated(0) / 1024**3,
    "model": BASE_MODEL_ID,
    "status": "T4_DRY_RUN_PASS",
    "execution_time_s": execution_time
}
report_path = os.path.join(BASE_DIR, 'reports', 'dry_run', f'{RUN_ID}_report.json')
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)


In [ ]:
# PHASE 5: SFT Authorization Gate
SFT_AUTH_TOKEN = "" # SET THIS TO A UNIQUE VALUE TO AUTHORIZE SFT

state = load_state()
if state["CURRENT_PHASE"] not in ["AWAITING_SFT_AUTHORIZATION", "SFT"]:
    raise RuntimeError(f"Cannot authorize SFT in phase: {state['CURRENT_PHASE']}")

if not SFT_AUTH_TOKEN:
    raise RuntimeError("SFT Authorization required! Provide a fresh token to SFT_AUTH_TOKEN and re-run.")

if consume_authorization(SFT_AUTH_TOKEN) and state["CURRENT_PHASE"] == "AWAITING_SFT_AUTHORIZATION":
    raise RuntimeError("This authorization token has already been consumed.")

if state["CURRENT_PHASE"] == "AWAITING_SFT_AUTHORIZATION":
    add_authorization(SFT_AUTH_TOKEN)
    set_phase("SFT")

print("SFT AUTHORIZED.")


In [ ]:
# PHASE 6: SFT Training
from trl import SFTTrainer, SFTConfig

if load_state()["CURRENT_PHASE"] != "SFT":
    raise RuntimeError("Not authorized for SFT.")

sft_checkpoint_dir = os.path.join(BASE_DIR, 'checkpoints', 'sft', RUN_ID)
if os.path.exists(sft_checkpoint_dir):
    print(f"SFT checkpoint for {RUN_ID} already exists! Skipping training to prevent overwrite.")
else:
    print("Re-verifying hashes before training...")
    if get_hash(SFT_PATH) != EXPECTED_SFT_HASH:
        raise RuntimeError("Hash mutated before SFT!")

    sft_config = SFTConfig(
        output_dir=os.path.join(BASE_DIR, "outputs", "sft_temp"),
        learning_rate=2e-4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=3,
        max_seq_length=4096,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        seed=42,
        dataset_text_field="text",
        fp16=True
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        peft_config=lora_config,
        tokenizer=tokenizer,
    )

    print("Starting SFT Training...")
    trainer.train()
    
    os.makedirs(sft_checkpoint_dir, exist_ok=True)
    trainer.save_model(sft_checkpoint_dir)
    print(f"SFT Adapter saved to {sft_checkpoint_dir}")


In [ ]:
# PHASE 7: SFT Evaluation
print("Evaluating BASE vs SFT...")
# (Evaluation logic would run here)

set_phase("AWAITING_DPO_AUTHORIZATION")
print("STATUS: READY_FOR_DPO_AUTHORIZATION")


In [ ]:
# PHASE 8: DPO Authorization Gate
DPO_AUTH_TOKEN = "" # SET THIS TO A UNIQUE VALUE TO AUTHORIZE DPO

state = load_state()
if state["CURRENT_PHASE"] not in ["AWAITING_DPO_AUTHORIZATION", "DPO"]:
    raise RuntimeError(f"Cannot authorize DPO in phase: {state['CURRENT_PHASE']}")

if not DPO_AUTH_TOKEN:
    raise RuntimeError("DPO Authorization required! Provide a fresh token to DPO_AUTH_TOKEN and re-run.")

if consume_authorization(DPO_AUTH_TOKEN) and state["CURRENT_PHASE"] == "AWAITING_DPO_AUTHORIZATION":
    raise RuntimeError("This authorization token has already been consumed.")

if state["CURRENT_PHASE"] == "AWAITING_DPO_AUTHORIZATION":
    add_authorization(DPO_AUTH_TOKEN)
    set_phase("DPO")

print("DPO AUTHORIZED.")


In [ ]:
# PHASE 9: DPO Training
from trl import DPOTrainer, DPOConfig

if load_state()["CURRENT_PHASE"] != "DPO":
    raise RuntimeError("Not authorized for DPO.")

dpo_checkpoint_dir = os.path.join(BASE_DIR, 'checkpoints', 'dpo', RUN_ID)
if os.path.exists(dpo_checkpoint_dir):
    print(f"DPO checkpoint for {RUN_ID} already exists! Skipping training to prevent overwrite.")
else:
    print("Re-verifying hashes before training...")
    if get_hash(DPO_PATH) != EXPECTED_DPO_HASH:
        raise RuntimeError("Hash mutated before DPO!")

    dpo_dataset = load_dataset("json", data_files=DPO_PATH, split="train")

    dpo_config = DPOConfig(
        output_dir=os.path.join(BASE_DIR, "outputs", "dpo_temp"),
        learning_rate=5e-7,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        max_prompt_length=2048,
        max_length=4096,
        beta=0.1,
        optim="paged_adamw_8bit",
        fp16=True,
        gradient_checkpointing=True
    )

    dpo_trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=dpo_dataset,
        tokenizer=tokenizer,
        peft_config=lora_config,
    )

    print("Starting DPO Training...")
    dpo_trainer.train()
    
    os.makedirs(dpo_checkpoint_dir, exist_ok=True)
    dpo_trainer.save_model(dpo_checkpoint_dir)
    print(f"DPO Adapter saved to {dpo_checkpoint_dir}")


In [ ]:
# PHASE 10 & 11: Final Evaluation & Report
print("Evaluating BASE vs SFT vs DPO...")
# (Final evaluation logic)

set_phase("COMPLETE")
print("Workflow complete.")
